# Lezione 3 — Train, validation e test: valutare senza barare

**Come si usa questo notebook.** Come le lezioni precedenti: leggi dall'alto in
basso, esegui ogni cella, prova l'esercizio prima di aprire la soluzione, e
alla fine fai crescere il progetto. Prerequisito: aver eseguito le Lezioni 1 e 2.

**Cosa saprai fare alla fine:** davanti a *qualsiasi* problema predittivo —
un modello, ma anche una semplice regola decisionale — saprai stimare
onestamente quanto funzionerà su dati che non ha mai visto, scegliendo il
tipo di divisione giusto per la natura dei dati.

## Parte 1 — Teoria: memorizzare non è imparare

La domanda fondamentale di tutto il machine learning è questa: **come faccio a
sapere se una regola funziona su dati che non ha mai visto?**

Pensa a uno studente che prepara un esame studiando *le soluzioni degli esami
degli anni scorsi a memoria*. Sui vecchi esami fa punteggio pieno. All'esame
vero, con domande nuove, crolla. Non aveva *imparato* la materia: l'aveva
*memorizzata*. I modelli fanno esattamente lo stesso, e lo fanno benissimo —
un modello abbastanza flessibile può "memorizzare" qualsiasi tabella.

Da qui la regola che non ha eccezioni: **la valutazione si fa su dati che il
modello non ha mai toccato**. Se valuti sugli stessi dati usati per costruire
la regola, il punteggio misura la memoria, non la capacità di generalizzare —
e nel mondo reale ti accorgi dell'errore solo quando il sistema è già in
produzione.

Questo vale anche fuori dal machine learning: una strategia di trading
"testata" sugli stessi anni su cui è stata progettata, una regola aziendale
tarata sui casi già visti. Il principio è identico ovunque.

## Parte 2 — Teoria: perché tre insiemi e non due

La divisione standard è in **tre** insiemi, ognuno con un contratto preciso:

| Insieme | A cosa serve | Quante volte lo usi |
|---|---|---|
| **Train** | far imparare la regola | in continuazione |
| **Validation** | confrontare alternative e regolare le scelte | molte volte |
| **Test** | la stima finale e onesta | **una volta sola, alla fine** |

Perché non bastano train e test? Perché durante lo sviluppo farai decine di
scelte: quale modello, quali feature, quali parametri. Se ogni scelta la fai
guardando il punteggio sul test, stai *ottimizzando sul test*: dopo venti
tentativi avrai scelto la combinazione che va bene **su quel** test per caso,
e la stima non è più onesta. Il validation set esiste per assorbire queste
scelte; il test resta sigillato fino alla fine.

Proporzioni tipiche: 70/15/15 oppure 80/10/10. Con pochi dati si usano
tecniche più raffinate (cross-validation, la vedrai più avanti), ma il
principio non cambia.

## Parte 3 — Teoria: *come* dividere dipende dai dati

Il modo di dividere non è un dettaglio tecnico: è una decisione che dipende
dalla natura dei dati.

1. **Divisione casuale** — la base, valida quando le righe sono indipendenti
   e senza ordine temporale. Sempre con un **seed** fissato: una divisione
   non riproducibile rende irripetibile ogni esperimento.

2. **Divisione stratificata** — quando c'è una classe rara, il caso puro può
   lasciarla fuori da un insieme. La stratificazione preserva le proporzioni
   delle classi in ogni insieme.

3. **Divisione temporale** — quando i dati hanno un ordine nel tempo, si
   addestra sul passato e si valuta sul futuro, **mai il contrario**: in
   produzione il futuro non lo conosci. Un modello addestrato con dati di
   luglio e valutato su maggio è una bugia metodologica.

Vediamo con gli occhi perché la stratificazione serve:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 40 casi, di cui solo 5 positivi (classe rara al 12.5%)
rng = np.random.default_rng(7)
dati = pd.DataFrame({'valore': rng.normal(0, 1, 40),
                     'classe': [1] * 5 + [0] * 35})

semplice_train, semplice_test = train_test_split(dati, test_size=0.25, random_state=3)
strat_train, strat_test = train_test_split(dati, test_size=0.25, random_state=3,
                                           stratify=dati['classe'])

print('Positivi nel test — divisione casuale    :', int(semplice_test['classe'].sum()), 'su', len(semplice_test))
print('Positivi nel test — divisione stratificata:', int(strat_test['classe'].sum()), 'su', len(strat_test))

Con la divisione casuale la classe rara può finire quasi tutta da una parte
(prova a cambiare `random_state` e osserva quanto balla); con quella
stratificata le proporzioni sono stabili per costruzione. Con dataset piccoli
o sbilanciati la stratificazione non è un'opzione, è igiene di base.

E la divisione temporale, visualizzata:

In [ ]:
import matplotlib.pyplot as plt

giorni = pd.date_range('2026-01-01', periods=100, freq='D')
n = len(giorni)
t_fine, v_fine = int(n * 0.7), int(n * 0.85)

fig, asse = plt.subplots(figsize=(10, 1.8))
asse.scatter(giorni[:t_fine], [0] * t_fine, s=12, label='train (passato)')
asse.scatter(giorni[t_fine:v_fine], [0] * (v_fine - t_fine), s=12, label='validation')
asse.scatter(giorni[v_fine:], [0] * (n - v_fine), s=12, label='test (futuro)')
asse.set_yticks([])
asse.legend(loc='upper left', ncol=3)
asse.set_title('Divisione temporale: si impara dal passato, si valuta sul futuro')
plt.tight_layout()
plt.show()

## Parte 4 — Esercizio guidato

Torniamo al dataset dei sensori della Lezione 1 (le 120 letture orarie).
Le letture hanno un ordine temporale, quindi la divisione giusta è quella
**temporale**: 70% train, 15% validation, 15% test.

Il tuo compito:

1. carica il dataset e **ordina** le righe per `recorded_at`;
2. calcola gli indici di taglio al 70% e all'85%;
3. crea i tre insiemi con lo slicing (`iloc`);
4. **verifica il contratto**: l'ultimo istante del train deve venire prima
   del primo istante del validation, e così via.

**Prova tu nella cella qui sotto.**

In [ ]:
from pathlib import Path

letture = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'environmental_sensor_missing_challenge.csv')

# Scrivi qui il tuo tentativo:
# 1) ordina per recorded_at (attenzione: convertilo con pd.to_datetime)
# 2) taglia al 70% e all'85%
# 3) verifica che i tempi non si sovrappongano

letture.head()

### Soluzione spiegata

- prima si converte `recorded_at` in datetime: ordinare stringhe funziona per
  fortuna con questo formato, ma è fragile — meglio ordinare tempi veri;
- le righe senza `recorded_at` non sono ordinabili: si scartano prima (è la
  policy dei campi critici della Lezione 1 che ritorna);
- gli `assert` finali sono il contratto: se un giorno qualcuno cambia la
  divisione e rompe l'ordine temporale, il notebook lo dice subito.

In [ ]:
ordinate = letture.dropna(subset=['recorded_at']).copy()
ordinate['recorded_at'] = pd.to_datetime(ordinate['recorded_at'])
ordinate = ordinate.sort_values('recorded_at').reset_index(drop=True)

n = len(ordinate)
fine_train, fine_val = int(n * 0.70), int(n * 0.85)

train = ordinate.iloc[:fine_train]
validation = ordinate.iloc[fine_train:fine_val]
test = ordinate.iloc[fine_val:]

assert train['recorded_at'].max() < validation['recorded_at'].min()
assert validation['recorded_at'].max() < test['recorded_at'].min()

print(f'train: {len(train)} letture, fino a {train.recorded_at.max()}')
print(f'val  : {len(validation)} letture, fino a {validation.recorded_at.max()}')
print(f'test : {len(test)} letture, fino a {test.recorded_at.max()}')

## Parte 5 — Il progetto: Memory AI Lab, passo 3

Colpo di scena realistico: il sistema è rimasto in funzione per **tre mesi
simulati** e ha accumulato uno storico di oltre 300 memorie
(`memory_events_history.csv`, generato con seed riproducibile). Prima di
poterci addestrare qualsiasi cosa serve:

1. far passare lo storico dalle **difese delle Lezioni 1 e 2** (che ora sono
   quattro righe di codice, perché i concetti li possiedi);
2. unirlo all'archivio esistente;
3. dividerlo **temporalmente** in train/validation/test — le memorie hanno
   timestamp, e il Memory AI Lab in produzione riceverà sempre memorie
   *future* rispetto a quelle su cui è stato costruito.

In [ ]:
archivio = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_events_clean.csv')
storico = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'memory_events_history.csv')
memorie = pd.concat([archivio, storico], ignore_index=True)
prima = len(memorie)

# Difese della Lezione 1: campi critici, poi type/importance recuperabili
memorie = memorie.dropna(subset=['memory_id', 'text', 'timestamp']).reset_index(drop=True)
memorie['type'] = memorie['type'].fillna('unknown')
numeri = pd.to_numeric(memorie['importance'], errors='coerce')

# Difese della Lezione 2: duplicati, tipi, contratto 0-1
testo_norm = memorie['text'].astype('string').str.strip().str.casefold()
doppi = (memorie.duplicated('memory_id', keep='first')
         | memorie.assign(_n=testo_norm).duplicated(['_n', 'timestamp'], keep='first'))
memorie = memorie.loc[~doppi].reset_index(drop=True)
numeri = numeri.loc[memorie.index]
memorie['importance'] = numeri.fillna(numeri[numeri.between(0, 1)].median()).clip(0, 1)

print(f'Ricevute {prima} memorie, conservate {len(memorie)} '
      f'(rimossi {int(doppi.sum())} duplicati)')

In [ ]:
memorie['timestamp'] = pd.to_datetime(memorie['timestamp'], format='mixed')
memorie = memorie.sort_values('timestamp').reset_index(drop=True)

n = len(memorie)
fine_train, fine_val = int(n * 0.70), int(n * 0.85)
mem_train = memorie.iloc[:fine_train]
mem_val = memorie.iloc[fine_train:fine_val]
mem_test = memorie.iloc[fine_val:]

assert mem_train['timestamp'].max() <= mem_val['timestamp'].min()
assert mem_val['timestamp'].max() <= mem_test['timestamp'].min()

processed = Path('..') / 'datasets' / 'processed'
mem_train.to_csv(processed / 'memory_train.csv', index=False)
mem_val.to_csv(processed / 'memory_val.csv', index=False)
mem_test.to_csv(processed / 'memory_test.csv', index=False)

riepilogo = pd.DataFrame({
    'train': mem_train['type'].value_counts(normalize=True).round(2),
    'validation': mem_val['type'].value_counts(normalize=True).round(2),
    'test': mem_test['type'].value_counts(normalize=True).round(2),
})
print(f'train {len(mem_train)} | val {len(mem_val)} | test {len(mem_test)}')
print('\nDistribuzione dei tipi per insieme (da tenere d\'occhio):')
riepilogo

Il progetto ora ha tre insiemi con un contratto temporale verificato dagli
assert, e la tabella delle proporzioni ti dice se i tipi di memoria sono
distribuiti in modo simile (con la divisione temporale non è garantito: se
noti squilibri forti, è un'informazione preziosa, non un errore).

## Cosa hai imparato

- Valutare sui dati di addestramento misura la **memoria**, non
  l'apprendimento: la valutazione onesta usa dati mai visti.
- **Tre insiemi, tre contratti**: train per imparare, validation per
  scegliere, test una volta sola alla fine.
- Ottimizzare guardando il test **consuma** il test.
- Il tipo di divisione segue la natura dei dati: casuale (con seed),
  stratificata (classi rare), **temporale** (dati con ordine nel tempo:
  mai addestrare sul futuro).

## Quiz

1. Hai provato 30 configurazioni scegliendo ogni volta quella col punteggio
   migliore sul test set. Il punteggio finale è onesto?
2. Perché per le memorie del progetto la divisione temporale è più corretta
   di quella casuale?
3. Nel dataset dei sensori, cosa sarebbe successo con una divisione casuale
   invece che temporale?

<details>
<summary><b>Apri le risposte</b></summary>

1. No: hai ottimizzato sul test. Dopo 30 tentativi, la configurazione scelta
   è quella che va bene *su quel* test anche per caso. Le 30 scelte andavano
   fatte sul validation; il test andava usato una sola volta, alla fine.
2. Perché in produzione il sistema riceverà memorie future rispetto a quelle
   su cui è stato costruito. La divisione temporale riproduce questa
   condizione; quella casuale mescolerebbe futuro nel train, gonfiando le
   metriche.
3. Letture di ore future sarebbero finite nel train e letture passate nel
   test: il "modello" avrebbe potuto sfruttare informazione dal futuro
   (temperature vicine nel tempo sono simili), producendo una stima
   ottimistica e falsa.
</details>

## Fonti

- scikit-learn, *Cross-validation: evaluating estimator performance*
  (concetti di train/test e valutazione):
  https://scikit-learn.org/stable/modules/cross_validation.html
- scikit-learn, `train_test_split` (parametri `stratify` e `random_state`):
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

## Prossima lezione

Hai diviso bene — ma dividere bene non basta. L'informazione può filtrare
dal futuro o dal target in modi molto più subdoli: il **data leakage**,
l'errore più costoso del machine learning. Lezione 4.